# TTA + komposisi record clipped — `Net3` notebook_8 tanpa diubah

Dua sumbu yang belum pernah disentuh di repo ini, dan **tidak satu pun** masuk
keluarga "model deep baru" yang sudah ditutup oleh 7 kegagalan:

1. **Test-time augmentation** atas grup simetri yang sama dengan augmentasi
   training (rotasi horizontal + flip vertikal). Mekanismenya pengurangan
   varians. Aman terhadap diagnostik transfer **secara konstruksi**.
2. **Bobot record `is_clipped` ×2,25** untuk menyamakan komposisi train (1,68%)
   dengan test (3,78%). Gap MAE-nya +0,050 bahkan setelah distribusi magnitudo
   distandarisasi.

Dua kali training menghasilkan **empat kandidat** karena TTA hanya menyentuh
inferensi, jadi tiap set bobot dievaluasi dua kali:

| | tanpa TTA | dengan TTA |
|---|---|---|
| A: resep nb8 apa adanya | `cnn12a` (kontrol, harus ≈ 0,2091) | `cnn12a_tta` |
| B: + bobot clipped ×2,25 | `cnn12b` | `cnn12b_tta` |

Matriks 2×2-nya lengkap, jadi efek TTA dan efek bobot clipped tetap terisolasi
meski satu notebook — justru lebih diagnostik daripada dua notebook terpisah.

## Aritmatika target — baca ini dulu

Kalibrasi dari 3 submission nyata (R2 0,92): **LB ≈ `lb_real` + 0,0082** (sd 0,0028).

| Target LB | butuh `lb_real` | Δ dari nb8 (0,1762) |
|---|---|---|
| 0,1837 (peringkat 1) | 0,1755 | −0,0007 |
| 0,1800 | 0,1718 | −0,0044 |
| 0,1750 | 0,1668 | −0,0094 |
| **0,1700** | **0,1618** | **−0,0144** |
| **0,1600** | **0,1518** | −0,0244 |

Terobosan terbesar yang pernah ditemukan di repo ini — CNN v3 di `notebook_8` — bernilai
**−0,0070**. Jadi **LB 0,17 butuh dua kali lipat terobosan terbesar itu**.

**LB 0,16 butuh `lb_real` 0,1518, dan itu di BAWAH batas oracle-grouping 0,1543** — yaitu
skor yang dicapai kalau kita *tahu persis* record mana milik event mana. Post-processing
tidak bisa menembusnya dengan model dasar sekarang; hanya model per-record yang jauh lebih
akurat yang bisa. Perlakukan **0,16 sebagai di luar jangkauan** pendekatan ini.

Karena itu tiap notebook mencetak verdikt eksplisit terhadap kelima ambang di atas.



---

## Kenapa batch ini ada, dan kenapa ia BUKAN "model deep baru"

Dua batch sebelumnya menjalankan 8 notebook GPU dan **7 gagal**. Tujuh arsitektur/regime
training berbeda — Inception, TCN dilatasi, konvolusi dalam, konvolusi+transformer,
augmentasi agresif, pra-latih SSL, kepala distribusi — semuanya mendarat di `lb_real`
**0,208–0,232** dengan korelasi **0,89–0,97** terhadap CNN v3. Kesimpulannya: itu plafon
yang bisa dicapai konvolusi 1D dari 5 detik waveform, dan **keluarga itu ditutup**.

Notebook ini tidak melanggar penutupan itu. `Net3` di sini **identik** dengan notebook_8 —
arsitektur, jadwal, augmentasi training, anchor ridge, semuanya. Yang diuji dua hal di luar
model:

### 1. Test-time augmentation (belum pernah dicoba sama sekali)

Kata "TTA" nol kemunculan di seluruh repo. CNN v3 dilatih dengan **rotasi horizontal acak**
(sudut kontinu pada pasangan kanal E/N) dan **flip vertikal** (kanal Z), karena orientasi
stasiun terhadap azimut sumber memang sembarang — itu simetri fisik yang sah. Tapi saat
inferensi model cuma dilihat satu orientasi, jadi ia hanya *kira-kira* invarian; sisa
ketidak-invarianannya masuk sebagai derau prediksi.

Merata-ratakan prediksi atas grup simetri itu menghilangkan derau tersebut **secara eksak**.
Ini bukan informasi baru dan bukan arsitektur baru — ini pengurangan varians, sumbu yang
belum pernah digarap. Ukuran efeknya bisa ditaksir dari sumbu varians yang sudah pernah
diukur: menambah 3 → 6 seed memberi CNN **−0,0017** (0,2091 → 0,2074), dengan korelasi
antar-paket 0,9875. TTA menyerang komponen varians yang **berbeda** dari seed.

Karena rotasi dan flip simetris, TTA **tidak bisa** menggeser mean prediksi subset
depth-bulat — jadi ia aman terhadap ketiga diagnostik transfer secara konstruksi. Itu
properti yang tidak dimiliki satu pun ide yang gagal di dua batch sebelumnya.

### 2. Bobot record `is_clipped` (diukur ulang, premisnya SUDAH dikoreksi)

Yang diukur di sesi ini pada OOF notebook_8:

| | train | test |
|---|---|---|
| proporsi `is_clipped` | 1,68% (n=176) | **3,78% (n=87)** |

| | MAE |
|---|---|
| record clipped | 0,1990 |
| record non-clipped | 0,1338 |
| **clipped, distandarisasi ke distribusi magnitudo non-clipped** | **0,1838** |

Standarisasi itu penting: record clipped magnitudonya memang lebih besar (5,65 vs 5,28),
jadi sebagian selisihnya bisa saja cuma komposisi. Ternyata bukan — **gap +0,050 bertahan**
setelah magnitudonya disamakan. Plafon teoretisnya `0,0378 × 0,050 ≈ 0,0019 lb_real`.

**Hipotesis yang SALAH dan sudah dibuang:** "kliping merusak amplitudo puncak, jadi latih
CNN dengan kliping sintetis." Diukur: record clipped asli di dataset ini cuma tersaturasi
**~5 dari 3000 sampel** (fraksi |a| ≥ 0,98·max: median 0,00167 vs 0,00133 di record normal).
Setara memotong di **c ≈ 0,90** — nyaris tidak mengubah waveform. Kliping sintetis akan jadi
augmentasi kosong. Ide itu dibatalkan sebelum menghabiskan kuota GPU.

**Yang tersisa dan diuji di sini:** subgrup ini **kurang terwakili 2,25×** relatif test.
Run B menaikkan bobot sampelnya ×2,25 di loss CNN (anchor ridge sengaja TIDAK diubah,
supaya efeknya terisolasi ke CNN).

> **Peringatan jujur soal run B.** *Importance weighting* komposisi secara global (record
> depth-bulat ×2,235) sudah pernah dicoba dan **gagal** — arahnya salah. Run B bentuknya
> sama, bedanya cuma sasarannya subgrup 1,68% alih-alih 32%, dan sasaran itu punya gap
> error terukur. Peluangnya sedang, bukan tinggi. Kalau gagal, **catat dan tutup**;
> jangan dicoba lagi dalam bentuk lain.

### Yang harus diabaikan di notebook ini

Baris **"gain sisi desimal"** dicetak sebagai informasi, **bukan kriteria seleksi**. Aturan
lama "gain sisi desimal harus negatif" sudah **DICABUT**: `notebook_10` dibangun untuk
memaksimalkannya dan LB-nya rugi 0,0022 (0,1849 → 0,1871). Sisi desimal sudah menyentuh
oracle (0,1437 vs ~0,1488).

### Kalibrasi angka LB — baca sebelum bersemangat

LB publik cuma **30% data test**. Diukur dengan bootstrap: sd selisih dua submission yang
korelasinya 0,999 adalah **0,0008**; untuk submission yang beda (corr 0,888) **0,0038**.
LB terbaik 0,1849, peringkat 1 di 0,1835 — selisih **0,0014**, yaitu **0,4 sigma**.
**Kita sudah efektif seri dengan peringkat 1.** Yang menentukan peringkat akhir adalah 70%
data yang belum terlihat, jadi kandidat dipilih dari generalisasi jujur (gain holdout,
menang n/12, diagnostik transfer) — **bukan** dengan mengoptimasi ke angka LB publik.


## Cara pakai

1. **Settings -> Accelerator -> GPU** (T4 atau P100).
2. Tambahkan kompetisinya lewat **Add Input**.
3. Upload folder `belum_run/ref/` sebagai **Kaggle Dataset**, lalu Add Input juga.
   Isinya `nb8_ref.npz` — prediksi 8 kandidat notebook_8. Tanpa file itu notebook tetap
   jalan, tapi hanya bisa melaporkan skor model tunggal dan **tidak bisa memberi verdikt
   terhadap target LB**.
4. Run All.
5. Sel terakhir membungkus semua hasil jadi **satu file zip** di `/kaggle/working/`.
   Unduh zip itu dari panel Output, ekstrak `.npz`-nya ke `Case 1/cache/`.

## Yang perlu dibaca sebelum percaya angka apapun

`eb_twmae` mentah **cacat** di sisi depth-bulat: rekonstruksi event lewat
`groupby(depth_km, magnitude)` melebur record depth-bulat jadi pseudo-event sampai
**401 record**, sehingga CV over-weight ekor sekitar **3x**. Notebook ini memakai
`lb_real`, bukan `eb_twmae`.

Dan `lb_real` sendiri mengukur **data train**. `bias_rnd` serta mean dan sd prediksi
subset depth-bulat mengukur **jarak ke test**. Kalau keduanya bertabrakan, **yang kedua
menang** — `notebook_2` dan `notebook_5` dua-duanya memperbaiki CV lalu memburuk di LB.

**Perkiraan waktu di GPU Kaggle (T4): ~3–3,5 jam (2 run × 15 fold; TTA menambah ~12 menit inferensi total).** Notebook mencetak sisa waktu setiap fold selesai, jadi progresnya kelihatan sejak fold pertama.


---
## 0. Setup, metrik, fitur

Semua kode inti disalin apa adanya dari `lab/` supaya angkanya sebanding dengan run lokal.

In [1]:
import os, sys, time, warnings, json, zipfile, glob
from pathlib import Path
import numpy as np, pandas as pd
from scipy import signal, stats
warnings.filterwarnings("ignore")

KG = "/kaggle/input/competitions/ai-innovation-case-task-1-seismic-magnitude-prediction/seismic-dataset"
DATA = Path(KG) if os.path.exists(KG) else Path("seismic-dataset")
OUT  = Path("/kaggle/working") if os.path.isdir("/kaggle/working") else Path("cache")
OUT.mkdir(exist_ok=True, parents=True)

# nb8_ref.npz = prediksi 8 kandidat notebook_8 (OOF + test) + bobot blend-nya.
# Diperlukan untuk memberi VERDIKT terhadap target LB. Upload folder `belum_run/ref/`
# sebagai Kaggle Dataset, atau taruh nb8_ref.npz di mana saja di bawah /kaggle/input.
# Lokasi yang sudah diketahui lebih dulu, baru penelusuran rekursif sebagai cadangan
# (kalau slug dataset-nya berubah, atau dijalankan di lokal).
REF_KNOWN = ["/kaggle/input/datasets/timotiuspatrick/nb8-ref/nb8_ref.npz",
             "/kaggle/input/nb8-ref/nb8_ref.npz"]
def _find_ref():
    for p in REF_KNOWN:
        if os.path.exists(p): return Path(p)
    for root in ("/kaggle/input", "/kaggle/working", ".", "..",
                 "kaggle/ref", "../kaggle/ref"):
        if not os.path.isdir(root): continue
        for dp, _, fs in os.walk(root):
            if "nb8_ref.npz" in fs: return Path(dp)/"nb8_ref.npz"
    return None
REFP = _find_ref()
print("DATA :", DATA, "| ada:", DATA.exists())
print("REF  :", REFP)
if REFP is None:
    print()
    print("!"*72)
    print("nb8_ref.npz TIDAK DITEMUKAN -- notebook 01 akan berhenti di sel Training.")
    print("!"*72)
    print("File ini (334 KB) berisi prediksi OOF + test 8 kandidat notebook_8 beserta")
    print("bobot blend-nya. Tanpa itu tidak ada acuan untuk dibandingkan, dan notebook 01")
    print("tidak punya apa-apa untuk dikoreksi.")
    print()
    print("Cara memasangnya:")
    print("  1. Di repo lokal, ambil  Case 1/belum_run/ref/nb8_ref.npz")
    print("  2. Kaggle -> Datasets -> New Dataset -> upload file itu -> Create")
    print("  3. Balik ke notebook ini -> panel kanan -> Input -> + Add Input")
    print("     -> tab Datasets -> cari dataset barusan -> tanda +")
    print("  4. Run All lagi. Baris REF di atas harus berubah jadi sebuah path.")
    print()
    print("Isi /kaggle/input sekarang:")
    if os.path.isdir("/kaggle/input"):
        for d in sorted(os.listdir("/kaggle/input")): print("   -", d)
    else:
        print("   (/kaggle/input tidak ada -- ini bukan lingkungan Kaggle)")
    print("!"*72)
import torch
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("DEV  :", DEV, torch.cuda.get_device_name(0) if DEV=="cuda" else "")
if DEV != "cuda":
    print("PERINGATAN: GPU tidak aktif. Settings -> Accelerator -> GPU.")


DATA : /kaggle/input/competitions/ai-innovation-case-task-1-seismic-magnitude-prediction/seismic-dataset | ada: True
REF  : /kaggle/input/datasets/timotiuspatrick/nb8-ref/nb8_ref.npz
DEV  : cuda Tesla T4


In [2]:
FS, P_IDX, EPS = 200, 200, 1e-15
train = (pd.read_csv(DATA/"train"/"train_meta.csv")
           .merge(pd.read_csv(DATA/"train"/"train_labels.csv"), on="sample_id"))
test  = pd.read_csv(DATA/"test"/"test_meta.csv")
y   = train.magnitude.values.astype(np.float64)
# Tidak ada event_id di dataset, padahal metriknya per-event. Rekonstruksi heuristik:
ev  = train.groupby(["depth_km","magnitude"]).ngroup().values
cnt = pd.Series(ev).map(pd.Series(ev).value_counts()).values

def mag_weight(m):
    m=np.asarray(m); return np.where(m<5.5,1.0,np.where(m<6.0,1.5,np.where(m<6.5,2.0,2.5)))
def eb_twmae(yt,yp,e):
    df=pd.DataFrame({"y":yt,"p":yp,"e":e}); df["err"]=np.abs(df.y-df.p)
    g=df.groupby("e").agg(mae=("err","mean"),m=("y","first"))
    w=mag_weight(g.m.values); return float((w*g.mae.values).sum()/w.sum())

SW   = mag_weight(y)/cnt
d_tr = train.depth_km.values; d_te = test.depth_km.values
rnd_tr = np.isclose(d_tr, np.round(d_tr)); rnd_te = np.isclose(d_te, np.round(d_te))
MR = rnd_tr & (d_tr < 100); MT = rnd_te & (d_te < 100)
FD = 0.49   # proporsi event depth-desimal di test

# ---- METRIK lb_real -------------------------------------------------------------
# eb_twmae mentah CACAT di sisi depth-bulat: groupby(depth_km, magnitude) melebur record
# depth-bulat jadi pseudo-event sampai 401 record, sehingga CV over-weight ekor ~3x.
# lb_real memecah pseudo-event depth-bulat DANGKAL jadi chunk seukuran event nyata.
# JANGAN pernah menilai kandidat dengan eb_twmae mentah.
def make_ev_real(seed, chunk=8):
    rng=np.random.RandomState(seed); e2=ev.copy(); nxt=ev.max()+1
    split_me = rnd_tr & (d_tr < 100)     # >=100 km: grup 67-100% murni, jangan dipecah
    for eid in np.unique(ev[split_me]):
        idx=np.where(ev==eid)[0]
        if len(idx) <= chunk*1.5: continue
        rng.shuffle(idx); k=max(2,int(round(len(idx)/chunk)))
        for j,part in enumerate(np.array_split(idx,k)):
            if j>0: e2[part]=nxt; nxt+=1
    return e2
EVR = [make_ev_real(s) for s in range(8)]
def part_real(p, dec):
    m=(~rnd_tr) if dec else rnd_tr
    return float(np.mean([eb_twmae(y[m],p[m],e[m]) for e in EVR]))
def lb_real(p):  return FD*part_real(p,True)+(1-FD)*part_real(p,False)
def bias_rnd(p): return float(p[MR].mean()-y[MR].mean())
def make_sw(cap=None, power=1.0):
    c=cnt.astype(float)
    if cap is not None: c=np.minimum(c,cap)
    return mag_weight(y)/(c**power)

# KALIBRASI lb_real -> LB, dari 3 titik submission nyata (R2 0,92):
#   nb3 0,1945 -> 0,1996 | nb6 0,1845 -> 0,1952 | nb8 0,1762 -> 0,1849
# Kalibrasi lb_real -> LB, 5 titik:
#   nb3 +0,0051 | nb6 +0,0107 | nb8 +0,0087 | nb9 +0,0138 (OUTLIER) | nb10 +0,0098
# Tanpa nb9: mean +0,0086, sd 0,0025 -- cukup rapat. nb9 satu-satunya kegagalan ARAH,
# dan sebabnya spesifik (set-transformer menyeragamkan prediksi dalam grup depth).
LB_OFFSET = 0.0086          # mean 4 titik tanpa outlier nb9, sd 0,0025
LB_JUARA  = 0.1762          # lb_real notebook_8 (LB 0,1849). Kandidat harus MENGALAHKAN ini.
def lb_hat(v): return v + LB_OFFSET
print("event efektif:", len(np.unique(ev)), "| record:", len(y))


event efektif: 905 | record: 10505


In [3]:
class Scorer:
    """eb_twmae versi bincount atas banyak realisasi event -- ~5000x lebih cepat."""
    def __init__(self, evs, keep):
        self.parts=[]
        for dec,f in ((True,FD),(False,1-FD)):
            m = keep & ((~rnd_tr) if dec else rnd_tr)
            idx=np.where(m)[0]; per=[]
            for e in evs:
                _,inv=np.unique(e[idx],return_inverse=True)
                nrec=np.bincount(inv).astype(float)
                mm=np.zeros(len(nrec)); mm[inv]=y[idx]
                w=mag_weight(mm); per.append((inv,nrec,w,w.sum(),len(nrec)))
            self.parts.append((idx,f,per))
        self.n=len(evs)
    def __call__(self, p):
        tot=0.0
        for idx,f,per in self.parts:
            err=np.abs(y[idx]-p[idx]); s=0.0
            for inv,nrec,w,wsum,ne in per:
                s+=(w*(np.bincount(inv,weights=err,minlength=ne)/nrec)).sum()/wsum
            tot+=f*s/self.n
        return tot

class Post:
    """smoothing per grup depth + ekspansi varians PER SUBSET.

    Ekspansi harus dipusatkan per subset: versi global mendorong subset depth-bulat
    (yang rata-ratanya di atas mean keseluruhan) ikut naik dan memperbesar bias_rnd.
    Memusatkan per subset memangkas bias_rnd +0,076 -> +0,047 dengan ongkos CV 0,003."""
    def __init__(self, dep, lam_dec=0.95, lam_deep=0.5, T_deep=150, a_dec=1.15, a_rnd=1.05):
        r=np.isclose(dep,np.round(dep))
        self.lam=np.where(~r, lam_dec, np.where(dep>=T_deep, lam_deep, 0.0))
        order=np.argsort(dep,kind="stable"); self.order=order
        ds=dep[order]; bnd=np.flatnonzero(np.diff(ds))+1
        self.slices=np.concatenate([[0],bnd,[len(ds)]])
        self.r=r; self.a_dec=a_dec; self.a_rnd=a_rnd
    def __call__(self, p):
        ps=p[self.order]; gm=np.empty_like(ps); sl=self.slices
        for i in range(len(sl)-1): gm[sl[i]:sl[i+1]]=np.median(ps[sl[i]:sl[i+1]])
        g=np.empty_like(p); g[self.order]=gm
        q=(1-self.lam)*p+self.lam*g; out=q.copy()
        for m,a in ((self.r,self.a_rnd),(~self.r,self.a_dec)):
            mu=q[m].mean(); out[m]=mu+a*(q[m]-mu)
        return out

class SideScorer:
    """eb_twmae di SATU sisi saja. Dipakai untuk DIAGNOSTIK KE-4 (uji sisi-desimal).

    notebook_9 lulus ketiga diagnostik transfer lama, `lb_real`-nya membaik -0,0044, ia
    menang 12/12 belahan -- dan LB-nya tetap MEMBURUK (0,1849 -> 0,1856). Sebabnya:
    `lb_real` memperbaiki BOBOT metrik tapi bukan LABEL-nya. Record dari gempa berbeda
    yang berbagi (depth, magnitude) tetap punya `y` identik, jadi metode apa pun yang
    menyeragamkan prediksi di dalam satu grup depth dibayar berlebihan di sisi bulat.

    Sisi desimal tidak punya masalah itu (event-nya nyata, 98% murni). Jadi: perubahan
    yang menang di sisi bulat tapi RUGI di sisi desimal adalah bendera merah."""
    def __init__(self, dec, keep):
        m = keep & ((~rnd_tr) if dec else rnd_tr)
        self.idx=np.where(m)[0]; self.per=[]
        for e in EVR:
            _,inv=np.unique(e[self.idx],return_inverse=True)
            nrec=np.bincount(inv).astype(float)
            mm=np.zeros(len(nrec)); mm[inv]=y[self.idx]
            w=mag_weight(mm); self.per.append((inv,nrec,w,w.sum(),len(nrec)))
    def __call__(self, p):
        err=np.abs(y[self.idx]-p[self.idx]); s=0.0
        for inv,nrec,w,wsum,ne in self.per:
            s+=(w*(np.bincount(inv,weights=err,minlength=ne)/nrec)).sum()/wsum
        return s/len(self.per)

sc      = Scorer(EVR, np.ones(len(y), bool))
post_tr = Post(d_tr); post_te = Post(d_te)


In [4]:
def _hp(a,f=0.075,order=2):
    b,aa=signal.butter(order,f/(FS/2),btype="high")
    return signal.filtfilt(b,aa,a,axis=1).astype(np.float32)
def L(x): return np.log10(np.abs(x)+EPS).astype(np.float32)
WINS=[0.25,0.5,1.0,1.5,2.0,3.0,4.0]

def extract(X, meta):
    t0=time.time(); out={}
    A=X.astype(np.float32); V=np.cumsum(_hp(A),axis=1)/FS; D=np.cumsum(_hp(V),axis=1)/FS
    def hv(W): return np.sqrt(W[:,:,0]**2+W[:,:,1]**2), np.abs(W[:,:,2])
    for nm,W in [("a",A),("v",V),("d",D)]:
        H,Vt=hv(W); allc=np.abs(W).max(axis=2)
        for s in WINS:
            n=int(s*FS)
            out[f"P{nm}_{s:g}"]=L(allc[:,P_IDX:P_IDX+n].max(1))
            out[f"P{nm}H_{s:g}"]=L(H[:,P_IDX:P_IDX+n].max(1))
            out[f"P{nm}V_{s:g}"]=L(Vt[:,P_IDX:P_IDX+n].max(1))
        out[f"pre_{nm}"]=L(allc[:,:P_IDX].max(1))
        out[f"snr_{nm}"]=out[f"P{nm}_4"]-out[f"pre_{nm}"]
        for s in [1.0,2.0,4.0]:
            n=int(s*FS)
            out[f"rms{nm}_{s:g}"]=L(np.sqrt((W[:,P_IDX:P_IDX+n,:]**2).mean(axis=(1,2))))
        out[f"hv_{nm}"]=L(H[:,P_IDX:].max(1))-L(Vt[:,P_IDX:].max(1))
    e=(A[:,P_IDX:,:]**2).sum(axis=2); ce=np.cumsum(e,axis=1); tot=ce[:,-1:]+EPS
    frac=ce/tot; out["logE"]=L(tot[:,0])
    for q in [0.1,0.25,0.5,0.75,0.9]:
        out[f"tE{int(q*100)}"]=(np.argmax(frac>=q,axis=1)/FS).astype(np.float32)
    out["E_slope"]=(out["tE90"]-out["tE10"]).astype(np.float32)
    for nm,W in [("a",A),("d",D)]:
        ab=np.abs(W[:,P_IDX:,:]).max(axis=2); tp=ab.argmax(axis=1)/FS
        out[f"tpk_{nm}"]=tp.astype(np.float32)
        out[f"rise_{nm}"]=(L(ab.max(1))/(tp+0.05)).astype(np.float32)
    n3=600
    num=(V[:,P_IDX:P_IDX+n3,:]**2).sum(1); den=(D[:,P_IDX:P_IDX+n3,:]**2).sum(1)
    out["tauc"]=L(2*np.pi*np.sqrt((den/(num+1e-30)).mean(1)))
    for s in [1.0,2.0]:
        n=int(s*FS); nu=(V[:,P_IDX:P_IDX+n,:]**2).sum(1)
        dv=np.gradient(V[:,P_IDX:P_IDX+n,:],1/FS,axis=1)
        out[f"taup_{s:g}"]=L(2*np.pi*np.sqrt((nu/((dv**2).sum(1)+1e-30)).mean(1)))
    for nm,W in [("a",A),("d",D)]:
        f,Pxx=signal.welch(W[:,P_IDX:,:],fs=FS,nperseg=256,axis=1)
        Pm=Pxx.mean(axis=2); tote=Pm.sum(1)+EPS
        for lo,hi in [(0.1,1),(1,3),(3,5),(5,10),(10,20),(20,50),(50,100)]:
            m=(f>=lo)&(f<hi)
            out[f"bd{nm}_{lo:g}_{hi:g}"]=L(Pm[:,m].sum(1))
            out[f"bdr{nm}_{lo:g}_{hi:g}"]=(Pm[:,m].sum(1)/tote).astype(np.float32)
        out[f"cent_{nm}"]=((Pm*f).sum(1)/tote).astype(np.float32)
        out[f"fdom_{nm}"]=f[Pm.argmax(1)].astype(np.float32)
        out[f"lhr_{nm}"]=L(Pm[:,(f>=0.1)&(f<3)].sum(1))-L(Pm[:,(f>=10)&(f<50)].sum(1))
    seg=A[:,P_IDX:,:]
    out["kurt"]=stats.kurtosis(seg,axis=1).mean(1).astype(np.float32)
    out["skew"]=np.abs(stats.skew(seg,axis=1)).mean(1).astype(np.float32)
    out["zcr"]=(np.diff(np.sign(seg),axis=1)!=0).mean(axis=(1,2)).astype(np.float32)
    Sg=seg-seg.mean(1,keepdims=True)
    evl=np.linalg.eigvalsh(np.einsum("nti,ntj->nij",Sg,Sg)/Sg.shape[1])
    evl=np.maximum(evl,0)+1e-30
    out["rectilin"]=(1-(evl[:,0]+evl[:,1])/(2*evl[:,2])).astype(np.float32)
    out["planarity"]=(1-2*evl[:,0]/(evl[:,1]+evl[:,2])).astype(np.float32)
    F=pd.DataFrame(out)
    dist=meta.distance_km.values.astype(np.float32); dep=meta.depth_km.values.astype(np.float32)
    F["distance_km"]=dist; F["depth_km"]=dep
    F["snr_db"]=meta.snr_db.values.astype(np.float32)
    F["is_clipped"]=meta.is_clipped.values.astype(np.int8)
    F["logdist"]=np.log10(dist+1); F["logdep"]=np.log10(dep+1)
    F["hypo"]=np.sqrt(dist**2+dep**2); F["loghypo"]=np.log10(F.hypo+1)
    for nm in ["a","v","d"]:
        for s in WINS:
            F[f"C{nm}_{s:g}"]=F[f"P{nm}_{s:g}"]+F["loghypo"]
            F[f"C{nm}d_{s:g}"]=F[f"P{nm}_{s:g}"]+1.5*F["logdist"]
        for s1,s2 in [(0.5,1.0),(1.0,2.0),(2.0,4.0),(0.5,4.0)]:
            F[f"g{nm}_{s1:g}_{s2:g}"]=F[f"P{nm}_{s2:g}"]-F[f"P{nm}_{s1:g}"]
    print("  extract %.0fs -> %d fitur"%(time.time()-t0,F.shape[1]))
    return F.astype(np.float32)

def build_input(X, meta):
    """Input CNN gaya MagNet: BENTUK gelombang dinormalisasi per-record, lalu skala
    amplitudo disuntikkan balik sebagai skalar.

    Catatan lama 'jangan normalisasi amplitudo per-record' benar untuk fitur pohon dan
    SALAH TOTAL untuk CNN -- waveform mentah membentang 5 orde magnitudo dan konvolusi
    tidak bisa belajar apapun dari itu. Aturan tanpa cakupan itu menahan model deep
    selama empat notebook."""
    A=X.astype(np.float32); V=np.cumsum(_hp(A),axis=1)/FS; D=np.cumsum(_hp(V),axis=1)/FS
    chans=[]; scales=[]
    for W in (A,V,D):
        mx=np.abs(W).max(axis=(1,2),keepdims=True)+1e-20
        chans.append((W/mx).transpose(0,2,1)); scales.append(np.log10(mx[:,0,0]))
    Wn=np.concatenate(chans,axis=1).astype(np.float32)
    dist=meta.distance_km.values.astype(np.float32); dep=meta.depth_km.values.astype(np.float32)
    S=np.stack([*scales, np.log10(dist+1), np.log10(dep+1),
                np.log10(np.sqrt(dist**2+dep**2)+1),
                meta.snr_db.values.astype(np.float32)/40.0,
                meta.is_clipped.values.astype(np.float32)],axis=1).astype(np.float32)
    return Wn, S

Xtr_raw = np.load(DATA/"train"/"train_waveforms.npy")
Xte_raw = np.load(DATA/"test"/"test_waveforms.npy")
t0=time.time()
Ftr = extract(Xtr_raw, train); Fte = extract(Xte_raw, test)
Wtr, Str = build_input(Xtr_raw, train)
Wte, Ste = build_input(Xte_raw, test)
del Xtr_raw, Xte_raw
print("siap %.0fs | Wtr %s  Str %s"%(time.time()-t0, Wtr.shape, Str.shape))


  extract 12s -> 196 fitur
  extract 3s -> 196 fitur
siap 18s | Wtr (10505, 9, 1000)  Str (10505, 8)


In [5]:
import torch.nn as nn
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import Ridge
A_BASE=["Cd_2","Cv_2","Ca_4","logdist","loghypo","logE","snr_db"]

class Blk(nn.Module):
    def __init__(s, ci, co, k=7, pool=4):
        super().__init__()
        s.c1=nn.Conv1d(ci,co,k,padding=k//2); s.b1=nn.BatchNorm1d(co)
        s.c2=nn.Conv1d(co,co,k,padding=k//2); s.b2=nn.BatchNorm1d(co)
        s.se=nn.Sequential(nn.AdaptiveAvgPool1d(1),nn.Conv1d(co,max(4,co//8),1),nn.SiLU(),
                           nn.Conv1d(max(4,co//8),co,1),nn.Sigmoid())
        s.sk=nn.Conv1d(ci,co,1) if ci!=co else nn.Identity()
        s.act=nn.SiLU(); s.pool=nn.MaxPool1d(pool)
    def forward(s,x):
        h=s.act(s.b1(s.c1(x))); h=s.b2(s.c2(h)); h=h*s.se(h)
        return s.pool(s.act(h+s.sk(x)))

class Net3(nn.Module):
    """CNN v3 notebook_8 (LB 0,1849). Acuan, bukan yang diuji di sini."""
    def __init__(s, n_scalar, w=64):
        super().__init__()
        s.body=nn.Sequential(Blk(9,w),Blk(w,w*2),Blk(w*2,w*3),Blk(w*3,w*4,k=5,pool=2),
                             Blk(w*4,w*4,k=3,pool=2))
        s.trunk=nn.Sequential(nn.Linear(w*8+n_scalar,256), nn.SiLU(), nn.Dropout(0.25),
                              nn.Linear(256,96), nn.SiLU())
        s.mag=nn.Linear(96,1); s.dep=nn.Linear(96,1)
    def forward(s,x,sc_):
        h=s.body(x); h=torch.cat([h.mean(-1),h.amax(-1),sc_],dim=1); h=s.trunk(h)
        return s.mag(h).squeeze(-1), s.dep(h).squeeze(-1)

def run(Wtr,Str,Wte,Ste, sw, seeds=(0,1,2), epochs=80, bs=128, lr=2.5e-3,
        avg_last=10, resid=True, arch=None):
    """Loop training CNN v3 persis seperti notebook_8.

    Empat hal yang membuat v3 menang: (1) kepala kedua memprediksi log10(depth), dihitung
    HANYA untuk record depth-desimal (di sana depth adalah inversi sungguhan, bukan default
    katalog); (2) augmentasi derau FISIK -- tempel jendela pre-P record lain, jadi derau
    stasiun sungguhan bukan Gaussian sintetis; (3) blok Squeeze-Excitation; (4) jadwal
    panjang + rata-rata bobot 10 epoch terakhir."""
    Xa=Ftr[A_BASE].values.astype(np.float64); Xa_te=Fte[A_BASE].values.astype(np.float64)
    dep_t=np.log10(d_tr+1).astype(np.float32); dep_ok=(~rnd_tr).astype(np.float32)
    noise=Wtr[:,:,:200].copy()
    oof=np.zeros(len(y)); pte=np.zeros(len(Wte))
    Xte_t=torch.tensor(Wte).to(DEV); Ste_t=torch.tensor(Ste).to(DEV)
    nf=0; t0=time.time()
    for seed in seeds:
        torch.manual_seed(seed); np.random.seed(seed)
        for ti,vi in GroupKFold(5).split(Wtr,y,ev):
            if resid:
                rg=Ridge(alpha=1.0).fit(Xa[ti],y[ti],sample_weight=sw[ti])
                tgt=(y-rg.predict(Xa)).astype(np.float32)
                off_v=rg.predict(Xa[vi]); off_t=rg.predict(Xa_te)
            else:
                tgt=y.astype(np.float32); off_v=0; off_t=0
            net=(arch or Net3)(Str.shape[1]).to(DEV)
            opt=torch.optim.AdamW(net.parameters(),lr=lr,weight_decay=2e-2)
            sch=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=lr,
                    total_steps=epochs*max(1,len(ti)//bs),pct_start=0.2)
            Xt=torch.tensor(Wtr[ti]); St=torch.tensor(Str[ti]); Nz=torch.tensor(noise[ti])
            tt=torch.tensor(tgt[ti]); wt=torch.tensor(sw[ti],dtype=torch.float32)
            dt=torch.tensor(dep_t[ti]); dk=torch.tensor(dep_ok[ti])
            Xv=torch.tensor(Wtr[vi]).to(DEV); Sv=torch.tensor(Str[vi]).to(DEV)
            av=np.zeros(len(vi)); at=np.zeros(len(Wte))
            for ep in range(epochs):
                net.train(); perm=torch.randperm(len(ti))
                for i in range(0,len(ti)-bs+1,bs):
                    b=perm[i:i+bs]; xb=Xt[b].clone(); sb=St[b].to(DEV)
                    th=torch.rand(len(b))*2*np.pi; cs,sn=torch.cos(th),torch.sin(th)
                    for k in (0,3,6):
                        e_,n_=xb[:,k].clone(),xb[:,k+1].clone()
                        xb[:,k]=cs[:,None]*e_-sn[:,None]*n_; xb[:,k+1]=sn[:,None]*e_+cs[:,None]*n_
                    if torch.rand(1).item()<0.5:
                        for k in (2,5,8): xb[:,k]*=-1
                    if torch.rand(1).item()<0.5:
                        j=torch.randint(0,len(ti),(len(b),))
                        amp=(torch.rand(len(b),1,1)*0.6)
                        xb=xb+amp*Nz[j].repeat(1,1,5)[:,:,:xb.shape[2]]
                    xb=xb.to(DEV)
                    pm,pdep=net(xb,sb)
                    lm=(wt[b].to(DEV)*(pm-tt[b].to(DEV)).abs()).sum()/wt[b].sum().to(DEV)
                    dkb=dk[b].to(DEV)
                    ld=((pdep-dt[b].to(DEV)).abs()*dkb).sum()/(dkb.sum()+1e-6)
                    opt.zero_grad(); (lm+0.15*ld).backward(); opt.step(); sch.step()
                if ep>=epochs-avg_last:
                    net.eval()
                    with torch.no_grad():
                        av+=net(Xv,Sv)[0].cpu().numpy()
                        at+=np.concatenate([net(Xte_t[i:i+512],Ste_t[i:i+512])[0].cpu().numpy()
                                            for i in range(0,len(Wte),512)])
            oof[vi]+=(av/avg_last+off_v)/len(seeds)
            pte+=(at/avg_last+off_t)/(5*len(seeds))
            nf+=1
            print("    fold %2d/%d selesai  (%.0f menit terpakai, ~%.0f menit tersisa)"
                  %(nf,5*len(seeds),(time.time()-t0)/60,
                    (time.time()-t0)/nf*(5*len(seeds)-nf)/60), flush=True)
    return oof,pte


In [6]:
REF = np.load(REFP, allow_pickle=True) if REFP else None
CAND = [str(x) for x in REF["__cand__"]] if REF is not None else []

def ref_stack():
    O=np.stack([REF[f"{n}__oof"].astype(np.float64) for n in CAND])
    E=np.stack([REF[f"{n}__pte"].astype(np.float64) for n in CAND])
    return O,E

# Validasi ref SEKARANG, bukan berjam-jam kemudian setelah training selesai.
# Kalau file-nya benar, blend 8 kandidat itu harus mereproduksi lb_real notebook_8.
if REF is not None:
    _o = post_tr(REF["__w__"] @ ref_stack()[0]); _v = sc(_o)
    print("ref termuat: %d kandidat -> lb_real %.4f  (nb8 seharusnya 0,1762)"%(len(CAND),_v))
    print("   " + ("COCOK -- verdikt bisa dipercaya." if abs(_v-0.1762) < 0.0005 else
                   "TIDAK COCOK -- file ref salah/rusak, verdikt tidak bisa dipercaya."))
    del _o, _v

def blend_search(names, Otr, Ote, ref_oof, nsamp=8000, alpha=0.6, seed=0):
    """Protokol nb6/nb8: cari bobot di separuh event, ukur di separuh LAIN, 12 belahan;
    bobot final = rata-rata lintas belahan (bukan argmin di data penuh).

    Ini perlu. Waktu MLP divergen (OOF 0,778) ikut jadi kandidat di nb5, pencarian bobot
    tetap memberinya 0,36 dan 'memperbaiki' skor. Jaga kandidat tetap sedikit."""
    rng=np.random.RandomState(seed)
    W=np.vstack([rng.dirichlet(np.ones(len(names))*alpha,size=nsamp),np.eye(len(names))])
    Q=np.stack([post_tr(x@Otr) for x in W])
    uev=np.unique(ev); gains=[];opt=[];ws=[];dgains=[]
    for rep in range(6):
        r=np.random.RandomState(100+rep); perm=r.permutation(uev)
        h1=np.isin(ev,perm[:len(perm)//2])
        for keep in (h1,~h1):
            ss=Scorer(EVR,keep); sh=Scorer(EVR,~keep); shd=SideScorer(True,~keep)
            v=np.array([ss(q) for q in Q]); i=v.argmin()
            gains.append(sh(Q[i])-sh(ref_oof)); opt.append(sh(Q[i])-v[i]); ws.append(W[i])
            dgains.append(shd(Q[i])-shd(ref_oof))     # DIAGNOSTIK 4: gain di sisi desimal
    wm=np.mean(ws,0); wm/=wm.sum()
    o=post_tr(wm@Otr); p=post_te(wm@Ote); om=float(np.mean(opt))/np.sqrt(2)
    return dict(w=wm, oof=o, pte=p, lb_real=float(sc(o)), jujur=float(sc(o))+om,
                optimisme=om, gain=float(np.mean(gains)), menang=int(sum(g<0 for g in gains)),
                bias=bias_rnd(o), dec=part_real(o,True), rnd=part_real(o,False),
                dec_gain=float(np.mean(dgains)),
                dec_menang=int(sum(g<0 for g in dgains)),
                mean_rnd_te=float(p[MT].mean()), sd_rnd_te=float(p[rnd_te].std()))

TARGET = [("peringkat 1 (Kingpul)", 0.1837), ("0,1800", 0.1800),
          ("0,1750", 0.1750), ("0,1700", 0.1700), ("0,1600", 0.1600)]

def verdikt(res, nama):
    """Cetak diagnostik transfer + verdikt target. Tiga diagnostik wajib sebelum submit."""
    est = lb_hat(res["lb_real"])
    print("="*70); print("VERDIKT -- %s"%nama); print("="*70)
    print("  lb_real        %.4f   (notebook_8 = 0,1762)"%res["lb_real"])
    print("  lb_real jujur  %.4f   (+optimisme seleksi %.4f)"%(res["jujur"],res["optimisme"]))
    print("  gain vs nb8    %+.4f  (menang %d/12 belahan)"%(res["gain"],res["menang"]))
    print("  dec %.4f  rnd %.4f   (nb8: 0,1437 / 0,2074)"%(res["dec"],res["rnd"]))
    print()
    print("  --- diagnostik transfer (SEMUA wajib aman sebelum submit) ---")
    ok1 = res["bias"] <= 0.05
    ok2 = abs(res["mean_rnd_te"]-5.272) <= 0.03
    ok3 = res["sd_rnd_te"] <= 0.42
    ok4 = res["lb_real"] < LB_JUARA
    print("  bias_rnd            %+.3f   %s  (tolak kalau > 0,05; nb8 +0,031)"
          %(res["bias"], "AMAN " if ok1 else "BAHAYA"))
    print("  mean test bulat<100 %.3f    %s  (optimum ~5,272; nb8 5,259)"
          %(res["mean_rnd_te"], "AMAN " if ok2 else "AWAS  "))
    print("  sd test sisi bulat  %.3f    %s  (naik = LB memburuk; nb8 0,396)"
          %(res["sd_rnd_te"], "AMAN " if ok3 else "AWAS  "))
    print("  lb_real vs juara    %.4f   %s  (harus < %.4f = notebook_8)"
          %(res["lb_real"], "AMAN " if ok4 else "BAHAYA", LB_JUARA))
    print()
    print("  --- informasi saja, BUKAN kriteria seleksi ---")
    print("  gain sisi desimal   %+.4f  (menang %d/12)"
          %(res.get("dec_gain",float("nan")), res.get("dec_menang",-1)))
    print("  Jangan pakai angka ini untuk menerima/menolak model per-record. Aturan lama")
    print("  'gain sisi desimal harus negatif' DICABUT: notebook_10 dibangun untuk")
    print("  memaksimalkannya dan LB-nya rugi 0,0022 (0,1849 -> 0,1871). Sisi desimal")
    print("  sudah menyentuh oracle (0,1437 vs ~0,1488); mengoptimalkannya cuma")
    print("  mengorbankan sisi bulat, yaitu 51% metrik dan seluruh headroom tersisa.")
    print()
    print("  ESTIMASI LB = lb_real + 0,0096 = %.4f   (sd 0,0036 -- TIDAK STABIL)"%est)
    print("  --- verdikt target ---")
    for nm,t in TARGET:
        d=est-t
        print("    %-22s %.4f   %s (selisih %+.4f)"
              %(nm,t,"TERCAPAI" if d<=0 else "belum   ",d))
    print("="*70)
    if not ok4:
        print("BAHAYA -- lb_real blend ini TIDAK mengalahkan notebook_8 (0,1762).")
        print("JANGAN disubmit. `notebook_10` melanggar aturan ini atas dasar teori dan")
        print("rugi 0,0022. Satu kegagalan LB bukan dasar cukup untuk membatalkan metrik")
        print("peringkat yang benar 4 dari 5 kali.")
    if not (ok1 and ok2 and ok3):
        print("PERINGATAN: diagnostik transfer tidak aman. `lb_real` mengukur data TRAIN;")
        print("`bias_rnd` dan mean subset mengukur JARAK KE TEST. Kalau bertabrakan, yang")
        print("kedua menang -- notebook_2 dan notebook_5 dua-duanya CV-nya membaik lalu")
        print("LB-nya memburuk karena aturan ini dilanggar.")
    return est

HASIL = []          # file yang dihasilkan notebook ini, untuk dibundel di akhir

def catat(p):
    p = str(p)
    if p not in HASIL: HASIL.append(p)
    return p

def simpan(nama, oof, pte, res=None):
    catat(OUT/f"{nama}.npz"); np.savez(OUT/f"{nama}.npz", oof=oof, pte=pte)
    print("tersimpan:", OUT/f"{nama}.npz")
    if res is not None:
        sub=pd.DataFrame({"sample_id":test.sample_id,"magnitude":np.round(res["pte"],3)})
        sub.to_csv(catat(OUT/f"{nama}_submission.csv"), index=False)
        json.dump({k:(np.asarray(v).tolist() if not np.isscalar(v) else float(v))
                   for k,v in res.items() if k not in ("oof","pte")},
                  open(catat(OUT/f"{nama}_score.json"),"w"), indent=2)
        print("tersimpan:", OUT/f"{nama}_submission.csv")

def bundel(nama_zip):
    """Bungkus hasil notebook INI jadi satu zip supaya gampang diunduh dari Kaggle.

    Hanya file yang dicatat lewat `catat()` yang ikut -- waveform dan input CNN yang
    berukuran ratusan MB sengaja ditinggal, karena yang dibutuhkan di lokal cuma .npz
    prediksi, .csv submission, dan .json skor."""
    z = OUT/f"{nama_zip}.zip"
    files = [f for f in HASIL if os.path.exists(f)]
    if not files:
        print("tidak ada file untuk dibundel."); return None
    with zipfile.ZipFile(z,"w",zipfile.ZIP_DEFLATED) as zf:
        for f in files: zf.write(f, arcname=os.path.basename(f))
    tot = sum(os.path.getsize(f) for f in files)/1024
    print("\n" + "="*70)
    print("BUNDEL SIAP DIUNDUH: %s   (%d file, %.0f KB)"%(z, len(files), tot))
    for f in files: print("   -", os.path.basename(f))
    print("="*70)
    print("Unduh zip itu dari panel Output di kanan, lalu ekstrak .npz-nya ke")
    print("`Case 1/cache/` supaya bisa dipakai blend berikutnya di lokal.")
    return z


ref termuat: 8 kandidat -> lb_real 0.1762  (nb8 seharusnya 0,1762)
   COCOK -- verdikt bisa dipercaya.


---
## 1. Grup simetri + loop training

`Net3` dan seluruh loop training disalin **apa adanya** dari notebook_8. Satu-satunya perubahan: saat evaluasi, prediksi dikumpulkan per elemen grup simetri alih-alih hanya orientasi kanonik. Karena disimpan per-elemen, sapuan K=1/2/4/8 keluar dari satu kali inferensi tanpa training ulang.

In [7]:
# ---------------------------------------------------------------------------------
# Grup simetri: SAMA PERSIS dengan yang dipakai augmentasi training notebook_8.
# Rotasi horizontal bekerja pada pasangan kanal (E,N) di tiap trio A/V/D -> indeks
# (0,1), (3,4), (6,7). Flip vertikal membalik kanal Z -> indeks 2, 5, 8.
#
# Urutan sengaja dibuat supaya PREFIX-nya bermakna, sehingga sapuan K bisa dilakukan
# gratis dari satu kali inferensi:
#   K=1  -> identitas saja (= tanpa TTA, dipakai sebagai kontrol)
#   K=2  -> + flip vertikal
#   K=4  -> + rotasi 180 derajat
#   K=8  -> + rotasi 90 dan 270
# ---------------------------------------------------------------------------------
SYMS = [(0.0, 1.0), (0.0, -1.0),
        (np.pi, 1.0), (np.pi, -1.0),
        (np.pi/2, 1.0), (np.pi/2, -1.0),
        (3*np.pi/2, 1.0), (3*np.pi/2, -1.0)]
K_SWEEP = [1, 2, 4, 8]

def apply_sym(x, ang, flip):
    """Terapkan satu elemen grup simetri pada batch bentuk (n, 9, T)."""
    x = x.clone()
    cs, sn = float(np.cos(ang)), float(np.sin(ang))
    if ang != 0.0:
        for k in (0, 3, 6):
            e_, n_ = x[:, k].clone(), x[:, k+1].clone()
            x[:, k] = cs*e_ - sn*n_
            x[:, k+1] = sn*e_ + cs*n_
    if flip < 0:
        for k in (2, 5, 8):
            x[:, k] = -x[:, k]
    return x

def predict_syms(net, Xt, St, bs=512):
    """Prediksi terpisah untuk TIAP elemen simetri -> array (len(SYMS), n).

    Disimpan per-elemen, bukan langsung dirata-ratakan, supaya K=1/2/4/8 bisa dibaca
    dari satu kali inferensi. Biaya tambahannya cuma inferensi, bukan training."""
    out = np.zeros((len(SYMS), len(Xt)))
    net.eval()
    with torch.no_grad():
        for j, (a, f) in enumerate(SYMS):
            acc = []
            for i in range(0, len(Xt), bs):
                acc.append(net(apply_sym(Xt[i:i+bs], a, f), St[i:i+bs])[0].cpu().numpy())
            out[j] = np.concatenate(acc)
    return out

def run_tta(Wtr, Str, Wte, Ste, sw, sw_loss=None, seeds=(0,1,2), epochs=80, bs=128,
            lr=2.5e-3, avg_last=10, tag=""):
    """`Net3` + loop training notebook_8 PERSIS, satu-satunya perubahan: saat evaluasi,
    prediksi dikumpulkan untuk tiap elemen grup simetri, bukan cuma orientasi kanonik.

    `sw`      -> bobot untuk anchor ridge (SELALU resep nb8, jangan diubah)
    `sw_loss` -> bobot untuk loss CNN (None = sama dengan `sw`)

    Memisahkan keduanya penting untuk run B: kalau anchor ikut berubah, gain/ruginya tidak
    bisa dipisahkan antara "CNN belajar lain" dan "baseline linearnya bergeser"."""
    if sw_loss is None:
        sw_loss = sw
    Xa = Ftr[A_BASE].values.astype(np.float64); Xa_te = Fte[A_BASE].values.astype(np.float64)
    dep_t = np.log10(d_tr+1).astype(np.float32); dep_ok = (~rnd_tr).astype(np.float32)
    noise = Wtr[:, :, :200].copy()
    oof_s = np.zeros((len(SYMS), len(y))); pte_s = np.zeros((len(SYMS), len(Wte)))
    Xte_t = torch.tensor(Wte).to(DEV); Ste_t = torch.tensor(Ste).to(DEV)
    nf = 0; t0 = time.time()
    for seed in seeds:
        torch.manual_seed(seed); np.random.seed(seed)
        for ti, vi in GroupKFold(5).split(Wtr, y, ev):
            # anchor ridge: target residual (trik nb5, dipakai nb8) -- pakai sw ASLI
            rg = Ridge(alpha=1.0).fit(Xa[ti], y[ti], sample_weight=sw[ti])
            tgt = (y - rg.predict(Xa)).astype(np.float32)
            off_v = rg.predict(Xa[vi]); off_t = rg.predict(Xa_te)
            net = Net3(Str.shape[1]).to(DEV)
            opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=2e-2)
            sch = torch.optim.lr_scheduler.OneCycleLR(
                opt, max_lr=lr, total_steps=epochs*max(1, len(ti)//bs), pct_start=0.2)
            Xt = torch.tensor(Wtr[ti]); St = torch.tensor(Str[ti]); Nz = torch.tensor(noise[ti])
            tt = torch.tensor(tgt[ti])
            wt = torch.tensor(sw_loss[ti], dtype=torch.float32)
            dt = torch.tensor(dep_t[ti]); dk = torch.tensor(dep_ok[ti])
            Xv = torch.tensor(Wtr[vi]).to(DEV); Sv = torch.tensor(Str[vi]).to(DEV)
            av = np.zeros((len(SYMS), len(vi))); at = np.zeros((len(SYMS), len(Wte)))
            for ep in range(epochs):
                net.train(); perm = torch.randperm(len(ti))
                for i in range(0, len(ti)-bs+1, bs):
                    b = perm[i:i+bs]; xb = Xt[b].clone(); sb = St[b].to(DEV)
                    th = torch.rand(len(b))*2*np.pi; cs, sn = torch.cos(th), torch.sin(th)
                    for k in (0, 3, 6):
                        e_, n_ = xb[:, k].clone(), xb[:, k+1].clone()
                        xb[:, k] = cs[:, None]*e_ - sn[:, None]*n_
                        xb[:, k+1] = sn[:, None]*e_ + cs[:, None]*n_
                    if torch.rand(1).item() < 0.5:
                        for k in (2, 5, 8): xb[:, k] *= -1
                    if torch.rand(1).item() < 0.5:
                        j = torch.randint(0, len(ti), (len(b),))
                        amp = (torch.rand(len(b), 1, 1)*0.6)
                        xb = xb + amp*Nz[j].repeat(1, 1, 5)[:, :, :xb.shape[2]]
                    xb = xb.to(DEV)
                    pm, pdep = net(xb, sb)
                    wb = wt[b].to(DEV)
                    lm = (wb*(pm - tt[b].to(DEV)).abs()).sum()/wb.sum()
                    dkb = dk[b].to(DEV)
                    ld = ((pdep - dt[b].to(DEV)).abs()*dkb).sum()/(dkb.sum()+1e-6)
                    opt.zero_grad(); (lm + 0.15*ld).backward(); opt.step(); sch.step()
                if ep >= epochs - avg_last:
                    av += predict_syms(net, Xv, Sv)
                    at += predict_syms(net, Xte_t, Ste_t)
            oof_s[:, vi] += (av/avg_last + off_v[None, :])/len(seeds)
            pte_s += (at/avg_last + off_t[None, :])/(5*len(seeds))
            nf += 1
            print("    %sfold %2d/%d selesai  (%.0f menit terpakai, ~%.0f menit tersisa)"
                  % (tag, nf, 5*len(seeds), (time.time()-t0)/60,
                     (time.time()-t0)/nf*(5*len(seeds)-nf)/60), flush=True)
    return oof_s, pte_s

def tta_k(oof_s, pte_s, k):
    """Rata-rata atas K elemen simetri pertama. k=1 berarti TANPA TTA."""
    return oof_s[:k].mean(0), pte_s[:k].mean(0)


In [8]:
def sapu_tta(oof_s, pte_s, nama):
    """Kurva TTA: apakah efeknya nyata dan apakah sudah jenuh di K=8.

    Kalau K=1 -> K=8 tidak menggerakkan angka, TTA ditutup sebagai keluarga dan tidak perlu
    dicoba lagi dengan K lebih besar. Kalau bergerak tapi belum datar di K=8, itu argumen
    untuk K=16 -- dan itu murah, karena tidak butuh training ulang."""
    rows = []
    for k in K_SWEEP:
        o, p = tta_k(oof_s, pte_s, k)
        po = post_tr(o)
        rows.append(dict(K=k, lb_real=round(lb_real(po), 4),
                         dec=round(part_real(po, True), 4),
                         rnd=round(part_real(po, False), 4),
                         bias_rnd=round(bias_rnd(po), 3),
                         mean_te=round(float(post_te(p)[MT].mean()), 3)))
    df = pd.DataFrame(rows)
    print("\n--- kurva TTA: %s ---" % nama)
    print(df.to_string(index=False))
    d = df.lb_real.iloc[0] - df.lb_real.iloc[-1]
    print("  K=1 -> K=8 : %+.4f  %s" % (-d, "BERGERAK" if abs(d) >= 0.0008 else "datar"))
    print("  bias_rnd bergeser %+.3f  (rotasi/flip simetris -> harusnya ~0)"
          % (df.bias_rnd.iloc[-1] - df.bias_rnd.iloc[0]))
    return df


---
## 2. Run A — kontrol: resep notebook_8 apa adanya

**Ini uji kewarasan lingkungan, bukan cuma baseline.** Resepnya identik `cnn3` (3 seed, 80 epoch, `sw = w(M)/sqrt(cnt)`), jadi `lb_real` di K=1 harus mendarat di **~0,2091**. Kalau tidak, ada yang beda di Kaggle dan **seluruh verdikt notebook ini tidak bisa dipercaya** — hentikan dan periksa dulu.

In [9]:
sw = make_sw(power=0.5)
t0 = time.time()
oofA_s, pteA_s = run_tta(Wtr, Str, Wte, Ste, sw, tag="A ")
np.savez(catat(OUT/"cnn12a_syms.npz"), oof_s=oofA_s, pte_s=pteA_s)
print("\nrun A selesai %.0f menit"%((time.time()-t0)/60))

dfA = sapu_tta(oofA_s, pteA_s, "A = resep nb8")
base = dfA.lb_real.iloc[0]
print("\nKONTROL: K=1 lb_real %.4f   (cnn3 di lokal = 0,2091)"%base)
print("  " + ("COCOK -- lingkungan Kaggle mereproduksi run lokal."
              if abs(base-0.2091) < 0.004 else
              "TIDAK COCOK -- selisih > 0,004 dari cnn3. Verdikt di bawah TIDAK\n  bisa dipercaya; periksa versi torch/CUDA dan data sebelum melanjutkan."))


    A fold  1/15 selesai  (5 menit terpakai, ~73 menit tersisa)
    A fold  2/15 selesai  (11 menit terpakai, ~69 menit tersisa)
    A fold  3/15 selesai  (16 menit terpakai, ~64 menit tersisa)
    A fold  4/15 selesai  (21 menit terpakai, ~58 menit tersisa)
    A fold  5/15 selesai  (27 menit terpakai, ~53 menit tersisa)
    A fold  6/15 selesai  (32 menit terpakai, ~48 menit tersisa)
    A fold  7/15 selesai  (37 menit terpakai, ~43 menit tersisa)
    A fold  8/15 selesai  (43 menit terpakai, ~37 menit tersisa)
    A fold  9/15 selesai  (48 menit terpakai, ~32 menit tersisa)
    A fold 10/15 selesai  (53 menit terpakai, ~27 menit tersisa)
    A fold 11/15 selesai  (58 menit terpakai, ~21 menit tersisa)
    A fold 12/15 selesai  (64 menit terpakai, ~16 menit tersisa)
    A fold 13/15 selesai  (69 menit terpakai, ~11 menit tersisa)
    A fold 14/15 selesai  (74 menit terpakai, ~5 menit tersisa)
    A fold 15/15 selesai  (80 menit terpakai, ~0 menit tersisa)

run A selesai 80 menit

---

---
## 3. Run B — bobot record `is_clipped` ×2,25

`sw_loss` dinaikkan ×2,25 khusus record clipped supaya bobot efektifnya di loss CNN sepadan dengan porsinya di test (1,68% → 3,78%). **Anchor ridge tetap memakai `sw` asli** — kalau anchor ikut bergeser, gain/ruginya tidak bisa dipisahkan antara "CNN belajar lain" dan "baseline linearnya bergeser".

> Ingat peringatannya: *importance weighting* komposisi global (depth-bulat ×2,235) sudah pernah gagal. Bedanya di sini sasarannya subgrup 1,68% dengan gap error terukur, bukan 32% populasi. Kalau gagal juga, **tutup keluarga ini**.

In [10]:
clip_tr = train.is_clipped.values.astype(bool)
clip_te = test.is_clipped.values.astype(bool)
rasio = clip_te.mean()/clip_tr.mean()
print("clipped: train %.2f%% (n=%d) | test %.2f%% (n=%d) | rasio %.2fx"
      %(clip_tr.mean()*100, clip_tr.sum(), clip_te.mean()*100, clip_te.sum(), rasio))
sw_b = sw.copy(); sw_b[clip_tr] *= rasio
print("bobot loss CNN: record clipped x%.2f  (anchor ridge TIDAK diubah)"%rasio)

t0 = time.time()
oofB_s, pteB_s = run_tta(Wtr, Str, Wte, Ste, sw, sw_loss=sw_b, tag="B ")
np.savez(catat(OUT/"cnn12b_syms.npz"), oof_s=oofB_s, pte_s=pteB_s)
print("\nrun B selesai %.0f menit"%((time.time()-t0)/60))

dfB = sapu_tta(oofB_s, pteB_s, "B = + bobot clipped")


clipped: train 1.68% (n=176) | test 3.78% (n=87) | rasio 2.26x
bobot loss CNN: record clipped x2.26  (anchor ridge TIDAK diubah)
    B fold  1/15 selesai  (5 menit terpakai, ~75 menit tersisa)
    B fold  2/15 selesai  (11 menit terpakai, ~69 menit tersisa)
    B fold  3/15 selesai  (16 menit terpakai, ~64 menit tersisa)
    B fold  4/15 selesai  (21 menit terpakai, ~58 menit tersisa)
    B fold  5/15 selesai  (27 menit terpakai, ~53 menit tersisa)
    B fold  6/15 selesai  (32 menit terpakai, ~48 menit tersisa)
    B fold  7/15 selesai  (37 menit terpakai, ~43 menit tersisa)
    B fold  8/15 selesai  (43 menit terpakai, ~37 menit tersisa)
    B fold  9/15 selesai  (48 menit terpakai, ~32 menit tersisa)
    B fold 10/15 selesai  (53 menit terpakai, ~27 menit tersisa)
    B fold 11/15 selesai  (59 menit terpakai, ~21 menit tersisa)
    B fold 12/15 selesai  (64 menit terpakai, ~16 menit tersisa)
    B fold 13/15 selesai  (69 menit terpakai, ~11 menit tersisa)
    B fold 14/15 selesai  (

---
## 4. Matriks 2×2 + MAE khusus record clipped

Dua pertanyaan terpisah, dan keduanya harus dijawab dari sini:

- **Apakah TTA menolong?** Bandingkan kolom (K=1 vs K=8) di dalam tiap baris.
- **Apakah bobot clipped menolong?** Bandingkan baris A vs B — dan yang paling   menentukan, lihat **MAE khusus record clipped**. Kalau run B tidak memperbaiki   angka itu, ia gagal pada mekanismenya sendiri dan `lb_real` yang kebetulan   membaik cuma derau.

In [11]:
kand = {}
rows = []
for nm, (o_s, p_s) in [("A", (oofA_s, pteA_s)), ("B", (oofB_s, pteB_s))]:
    for k, lbl in [(1, ""), (8, "_tta")]:
        o, p = tta_k(o_s, p_s, k)
        key = "cnn12%s%s" % (nm.lower(), lbl)
        kand[key] = (o, p)
        po = post_tr(o)
        e = np.abs(o - y)
        rows.append(dict(kandidat=key, K=k,
                         lb_real=round(lb_real(po), 4),
                         dec=round(part_real(po, True), 4),
                         rnd=round(part_real(po, False), 4),
                         bias_rnd=round(bias_rnd(po), 3),
                         MAE_clip=round(float(e[clip_tr].mean()), 4),
                         MAE_non=round(float(e[~clip_tr].mean()), 4)))
dfM = pd.DataFrame(rows)
print(dfM.to_string(index=False))
print("\npembanding notebook_8: cnn3_avg 0,2074 | cnn3_dir 0,2116 | LightGBM terbaik 0,2030")
print("MAE clipped pada blend nb8 (bukan CNN tunggal): 0,1990 vs non-clipped 0,1338")

gA = dfM[dfM.kandidat=="cnn12a"].lb_real.iloc[0] - dfM[dfM.kandidat=="cnn12a_tta"].lb_real.iloc[0]
gB = dfM[dfM.kandidat=="cnn12b"].lb_real.iloc[0] - dfM[dfM.kandidat=="cnn12b_tta"].lb_real.iloc[0]
gC = dfM[dfM.kandidat=="cnn12a"].lb_real.iloc[0] - dfM[dfM.kandidat=="cnn12b"].lb_real.iloc[0]
dC = dfM[dfM.kandidat=="cnn12a"].MAE_clip.iloc[0] - dfM[dfM.kandidat=="cnn12b"].MAE_clip.iloc[0]
print("\n--- efek terisolasi (positif = membaik) ---")
print("  TTA di run A            %+.4f"%gA)
print("  TTA di run B            %+.4f"%gB)
print("  bobot clipped (K=1)     %+.4f"%gC)
print("  bobot clipped: MAE record clipped %+.4f  <- mekanismenya sendiri"%dC)
if dC <= 0:
    print("  RUN B GAGAL PADA MEKANISMENYA. MAE record clipped tidak membaik, jadi")
    print("  perubahan lb_real apa pun di run B adalah derau. Tutup keluarga ini.")


  kandidat  K  lb_real    dec    rnd  bias_rnd  MAE_clip  MAE_non
    cnn12a  1   0.1947 0.1457 0.2417     0.032    0.3623   0.1821
cnn12a_tta  8   0.1936 0.1458 0.2396     0.032    0.3613   0.1812
    cnn12b  1   0.1914 0.1443 0.2366     0.038    0.3530   0.1810
cnn12b_tta  8   0.1904 0.1429 0.2359     0.037    0.3532   0.1800

pembanding notebook_8: cnn3_avg 0,2074 | cnn3_dir 0,2116 | LightGBM terbaik 0,2030
MAE clipped pada blend nb8 (bukan CNN tunggal): 0,1990 vs non-clipped 0,1338

--- efek terisolasi (positif = membaik) ---
  TTA di run A            +0.0011
  TTA di run B            +0.0010
  bobot clipped (K=1)     +0.0033
  bobot clipped: MAE record clipped +0.0093  <- mekanismenya sendiri


---
## 5. Blend — dua protokol, dan yang kedua yang lebih tepat

Semua batch sebelumnya cuma **menambahkan** kandidat ke-9 ke blend nb8. Untuk kandidat di sini itu protokol yang salah bentuknya: `cnn12a_tta` bukan model baru melainkan **versi lebih baik dari `cnn3_avg` yang sudah ada di dalam blend**. Menambahkannya sebagai kandidat ke-9 berarti blend memegang dua salinan model yang sama dengan korelasi ~0,99 — persis situasi yang membuat pencarian bobot menipu diri sendiri.

Jadi diuji dua-duanya:

- **tambah** — kandidat ke-9, seperti batch sebelumnya (untuk kesebandingan).
- **ganti** — `cnn3_avg` dikeluarkan, kandidat baru masuk menggantikannya.   Jumlah kandidat tetap 8, jadi tidak ada inflasi dari menambah derajat bebas.

Kriterianya tetap sama dan tidak dilonggarkan: **gain holdout negatif DAN menang ≥ 7/12 belahan DAN ketiga diagnostik transfer aman DAN `lb_real` < 0,1762.**

In [12]:
def uji(key, mode):
    """mode="tambah" -> kandidat ke-9 | mode="ganti" -> gantikan cnn3_avg."""
    o, p = kand[key]
    if mode == "ganti":
        keep = [i for i, n in enumerate(CAND) if n != "cnn3_avg"]
        names = [CAND[i] for i in keep] + [key]
        Otr = np.vstack([Oref[keep], o[None, :]])
        Ote = np.vstack([Eref[keep], p[None, :]])
    else:
        names = CAND + [key]
        Otr = np.vstack([Oref, o[None, :]])
        Ote = np.vstack([Eref, p[None, :]])
    r = blend_search(names, Otr, Ote, ref_oof)
    r["_nama"] = "%s (%s)" % (key, mode)
    r["_names"] = names
    return r

if REF is None:
    print("nb8_ref.npz tidak ditemukan -- verdikt blend dilewati.")
    hasil = []
else:
    Oref, Eref = ref_stack()
    ref_oof = post_tr(REF["__w__"] @ Oref)
    print("acuan notebook_8: lb_real=%.4f   [LB aktual 0,1849]\n"%sc(ref_oof))
    hasil = []
    for key in ["cnn12a_tta", "cnn12b_tta", "cnn12b"]:
        for mode in ["tambah", "ganti"]:
            r = uji(key, mode)
            hasil.append(r)
            print("%-24s lb_real %.4f  jujur %.4f  gain %+.5f  menang %2d/12  bias %+.3f"
                  %(r["_nama"], r["lb_real"], r["jujur"], r["gain"], r["menang"], r["bias"]))


acuan notebook_8: lb_real=0.1762   [LB aktual 0,1849]

cnn12a_tta (tambah)      lb_real 0.1759  jujur 0.1774  gain +0.00137  menang  1/12  bias +0.032
cnn12a_tta (ganti)       lb_real 0.1762  jujur 0.1774  gain +0.00142  menang  2/12  bias +0.034
cnn12b_tta (tambah)      lb_real 0.1753  jujur 0.1768  gain +0.00068  menang  3/12  bias +0.034
cnn12b_tta (ganti)       lb_real 0.1754  jujur 0.1766  gain +0.00044  menang  4/12  bias +0.035
cnn12b (tambah)          lb_real 0.1755  jujur 0.1769  gain +0.00077  menang  3/12  bias +0.034
cnn12b (ganti)           lb_real 0.1755  jujur 0.1766  gain +0.00052  menang  4/12  bias +0.035


---
## 6. Verdikt

Kandidat terbaik dipilih dari **gain holdout jujur**, bukan `lb_real` mentah — kandidat ke-N selalu bisa menurunkan angka di data tempat bobotnya dipilih (CNN+GRU menurunkan 0,1762 → 0,1756 tapi menang 1/12, dan menolaknya benar).

In [13]:
if hasil:
    best = min(hasil, key=lambda r: r["gain"])
    print("kandidat terbaik menurut gain holdout: %s\n"%best["_nama"])
    print("bobot:", dict(zip(best["_names"], np.round(best["w"], 3))))
    print("bobot model baru: %.3f\n"%best["w"][-1])
    est = verdikt(best, best["_nama"])
    simpan("blend_cnn12", best["oof"], best["pte"], best)
    lolos = (best["gain"] < 0 and best["menang"] >= 7 and best["bias"] <= 0.05
             and best["lb_real"] < LB_JUARA)
    print()
    print("LOLOS PROTOKOL -- layak jadi kandidat submission." if lolos else
          "TIDAK LOLOS -- notebook_8 tetap submission terbaik. Catat kegagalannya\n  di sudah_run/README.md beserta ALASANNYA; yang gagal justru yang paling sering\n  dicoba ulang kalau tidak dicatat.")
    print()
    print("Ingat: hanya ADA SATU submission final. Selisih ke peringkat 1 (0,0014)")
    print("ada di dalam derau LB publik (sd selisih ~0,004 untuk model berbeda),")
    print("jadi kandidat yang cuma menang tipis di lb_real BUKAN alasan cukup untuk")
    print("menggeser notebook_8. Ambangnya harus gain holdout jujur, bukan angka LB.")


kandidat terbaik menurut gain holdout: cnn12b_tta (ganti)

bobot: {'swA_sqrt': np.float64(0.043), 'resA_dep': np.float64(0.09), 'resB_dep': np.float64(0.163), 'resB_c30': np.float64(0.129), 'resB_p05': np.float64(0.044), 'cnnres': np.float64(0.01), 'cnn3_dir': np.float64(0.217), 'cnn12b_tta': np.float64(0.304)}
bobot model baru: 0.304

VERDIKT -- cnn12b_tta (ganti)
  lb_real        0.1754   (notebook_8 = 0,1762)
  lb_real jujur  0.1766   (+optimisme seleksi 0.0012)
  gain vs nb8    +0.0004  (menang 4/12 belahan)
  dec 0.1425  rnd 0.2070   (nb8: 0,1437 / 0,2074)

  --- diagnostik transfer (SEMUA wajib aman sebelum submit) ---
  bias_rnd            +0.035   AMAN   (tolak kalau > 0,05; nb8 +0,031)
  mean test bulat<100 5.259    AMAN   (optimum ~5,272; nb8 5,259)
  sd test sisi bulat  0.396    AMAN   (naik = LB memburuk; nb8 0,396)
  lb_real vs juara    0.1754   AMAN   (harus < 0.1762 = notebook_8)

  --- informasi saja, BUKAN kriteria seleksi ---
  gain sisi desimal   -0.0003  (menang 8/1

ValueError: could not convert string to float: 'cnn12b_tta (ganti)'

---
## 7. Simpan kandidat + bungkus jadi satu zip

Keempat kandidat disimpan supaya blend gabungan bisa diuji ulang di lokal — pertanyaan yang tidak bisa dijawab notebook ini sendirian.

In [ ]:
for key, (o, p) in kand.items():
    simpan(key, o, p)
bundel("hasil_cnn12_tta")
